Generar datos sinteitos relacions para llegar a 10000 registros

In [1]:
import pandas as pd
import numpy as np
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

In [2]:
# 1. Cargar tus datos (ya corregidos)
df = pd.read_csv('../data/raw/dataset.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [5]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [28]:
# 2. Detectar automáticamente la estructura (Metadata)
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

# Marcamos el customerID como clave primaria para que genere valores únicos
metadata.update_column(column_name='customerID', sdtype='id')

# 3. Entrenar el modelo (CTGAN es ideal para este tipo de datos mixtos)
synthesizer = CTGANSynthesizer(metadata)
synthesizer.fit(df)

# 4. Generar 3000 nuevos clientes sintéticos
new_data = synthesizer.sample(num_rows=3000)

# Guardar el resultado
new_data.to_csv('../data/raw/clientes_sinteticos_telco.csv', index=False)
print("¡Datos sintéticos generados con éxito!")

/home/danilavia/.local/lib/python3.14/site-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/danilavia/.local/lib/python3.14/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


¡Datos sintéticos generados con éxito!


In [16]:
df1 = pd.read_csv('../data/raw/clientes_sinteticos_telco.csv')
df_export = pd.concat([df, df1], axis=0)


In [17]:
#EXPORTARMOS LOS DATOS A CSV
df_export.to_csv('../data/raw/dataset_revision.csv', index=False)

In [14]:
# 2. Detectar automáticamente la estructura (Metadata)
metadata_nuevo = SingleTableMetadata()
metadata_nuevo.detect_from_dataframe(df)

# Marcamos el customerID como clave primaria para que genere valores únicos
metadata_nuevo.update_column(column_name='customerID', sdtype='id')

# 3. Entrenar el modelo (CTGAN es ideal para este tipo de datos mixtos)
synthesizer = CTGANSynthesizer(metadata_nuevo)
synthesizer.fit(df)

# 4. Generar 3000 nuevos clientes sintéticos
new_data_prueba = synthesizer.sample(num_rows=3000)



/home/danilavia/.local/lib/python3.14/site-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/danilavia/.local/lib/python3.14/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [ ]:
new_data_prueba = new_data_prueba.drop(['customerID'], axis=1)
new_data_prueba = new_data_prueba.drop(['Churn'], axis=1)
new_data_prueba.rename(columns={'tenure': 'Antiguedad'}, inplace=True)


In [ ]:
# Guardar el dataframe en csv para luego predecir resultados
new_data_prueba.to_csv('../data/processed/datos_prueba_ml.csv', index=False)
print("¡Datos sintéticos generados con éxito!")

¡Datos sintéticos generados con éxito!
